In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed

# -----------------------------
# 1. Amino acid encoder
# -----------------------------
def build_aa_encoder():
    aa_list = list("ACDEFGHIKLMNPQRSTVWY")
    extras = ["X", "B", "Z", "J", "U", "O", "-", "*", "?"]
    table = {aa: i for i, aa in enumerate(aa_list)}
    for k, aa in enumerate(extras, start=len(table)):
        table[aa] = k
    return table

AA_TO_INT = build_aa_encoder()

def encode_sequences_to_uint8(seqs, L_expected=None):
    if L_expected is not None:
        for s in seqs:
            if len(s) != L_expected:
                raise ValueError(f"Expected length {L_expected}, got {len(s)}")
    L = len(seqs[0])
    arr = np.empty((len(seqs), L), dtype=np.uint8)
    for i, s in enumerate(seqs):
        row = np.fromiter((AA_TO_INT.get(ch, 255) for ch in s), count=L, dtype=np.uint16)
        arr[i] = np.minimum(row, 255).astype(np.uint8)
    return arr

# -----------------------------
# 2. Pairwise distances (parallel)
# -----------------------------
def _compute_block(i, arr, ids):
    """Compute distances for a single i against all j > i."""
    a = arr[i]
    results = []
    for j in range(i+1, len(ids)):
        dist = np.sum(a != arr[j])
        # canonicalize ordering
        id1, id2 = (ids[i], ids[j]) if ids[i] < ids[j] else (ids[j], ids[i])
        results.append((id1, id2, int(dist)))
    return results

def compute_pairwise_df(df, L_expected=None, n_jobs=16):
    ids = df["Sequence_ID"].astype(str).to_numpy()
    seqs = df["Sequence"].astype(str).to_list()
    
    N = len(ids)
    arr = encode_sequences_to_uint8(seqs, L_expected=L_expected)
    
    # parallel outer loop
    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_compute_block)(i, arr, ids) for i in tqdm(range(N), desc="Computing pairwise distances")
    )
    
    # flatten results
    flat_results = [item for sublist in results for item in sublist]
    
    return pd.DataFrame(flat_results, columns=["Seq_1", "Seq_2", "Distance"])


In [2]:
date = "2021-07-01"
current_data = pd.read_csv(f"/data/sukhanna/MPP/ver_2/ver_2A/new_MAT/Disjoint_30D/{date}/current.csv")
dist_df = compute_pairwise_df(current_data, L_expected=1273)

Computing pairwise distances: 100%|█████████████████████████████████████████████████| 17726/17726 [02:20<00:00, 126.25it/s]


In [3]:
# Save to parquet
dist_df.to_parquet(f"/home/pgangwar@AD.UCSD.EDU/FutureCoV/{date}/pairwise.parquet", engine="pyarrow", compression="snappy", index=False)

In [7]:
print(date)

2021-07-01
